# Basic Training Walkthrough

This notebook demonstrates the basic training pipeline for class-wise autoencoders
implementing the RER (Reconstruction Error Ratio) framework from Marks et al. (2024).

## Overview

1. Extract features using DINOv2 or CLIP
2. Train class-wise autoencoders
3. Compute RER for dataset difficulty estimation

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import numpy as np

from model.ShallowAutoencoder import ShallowAutoencoder
from model.ClasswiseAutoencoderManager import ClasswiseAutoencoderManager
from task.ClasswiseAETask import ClasswiseAETask
from util.rer_metrics import compute_rer, compute_rer_statistics

## 1. Create Sample Data

For this demo, we'll create synthetic features. In practice, you would:
1. Run `scripts/extract_features.py` to extract DINOv2/CLIP features
2. Use `CachedFeatureDatamodule` to load the cached features

In [ ]:
# Simulate extracted features (in practice, use CachedFeatureDatamodule)
num_samples = 1000
num_classes = 10
feature_dim = 384  # DINOv2 ViT-S dimension

# Create features with some class structure
torch.manual_seed(42)
features = torch.randn(num_samples, feature_dim)
labels = torch.randint(0, num_classes, (num_samples,))

# Add class-specific bias to make classification meaningful
for c in range(num_classes):
    mask = labels == c
    features[mask] += torch.randn(feature_dim) * 0.5

# Normalize to [0, 1] for autoencoder
features = (features - features.min()) / (features.max() - features.min())

print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Class distribution: {torch.bincount(labels)}")

## 2. Create the Class-wise Autoencoder Manager

In [ ]:
# Create the manager with K=10 autoencoders
manager = ClasswiseAutoencoderManager(
    num_classes=num_classes,
    feature_dim=feature_dim,
    n_components=10,
    hidden_dims=[256],
    dropout=0.01,
    l2_reg=1e-6,
)

print(f"Number of autoencoders: {len(manager.autoencoders)}")
print(f"Parameters per AE: {sum(p.numel() for p in manager.autoencoders[0].parameters()):,}")

## 3. Train the Autoencoders

In [ ]:
# Training loop
optimizer = torch.optim.Adam(manager.parameters(), lr=0.001)
num_epochs = 50
batch_size = 64

losses = []

for epoch in range(num_epochs):
    # Shuffle data
    perm = torch.randperm(num_samples)
    features_shuffled = features[perm]
    labels_shuffled = labels[perm]
    
    epoch_loss = 0.0
    num_batches = 0
    
    for i in range(0, num_samples, batch_size):
        batch_features = features_shuffled[i:i+batch_size]
        batch_labels = labels_shuffled[i:i+batch_size]
        
        optimizer.zero_grad()
        loss, _ = manager.compute_total_loss(batch_features, batch_labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## 4. Compute RER Statistics

In [ ]:
# Compute reconstruction errors from all autoencoders
manager.eval()
with torch.no_grad():
    all_errors = manager.compute_all_reconstruction_errors(features)
    rer_values = compute_rer(all_errors, labels)

# Compute statistics
stats = compute_rer_statistics(rer_values, labels)

print("\nRER Statistics:")
print(f"  Chi (mean RER): {stats['chi']:.4f}")
print(f"  RER std: {stats['rer_std']:.4f}")
print(f"  RER range: [{stats['rer_min']:.4f}, {stats['rer_max']:.4f}]")
print(f"  Fraction above 1: {stats['frac_above_1']:.2%}")
print(f"\n  Hardest class: {stats['hardest_class']} (chi={stats['hardest_class_chi']:.4f})")
print(f"  Easiest class: {stats['easiest_class']} (chi={stats['easiest_class_chi']:.4f})")

In [ ]:
# Plot RER distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall RER histogram
axes[0].hist(rer_values.numpy(), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=1.0, color='r', linestyle='--', label='RER=1 (decision boundary)')
axes[0].axvline(x=stats['chi'], color='g', linestyle='-', label=f'Chi={stats["chi"]:.2f}')
axes[0].set_xlabel('RER')
axes[0].set_ylabel('Count')
axes[0].set_title('RER Distribution')
axes[0].legend()

# Per-class chi
class_chis = [stats.get(f'chi_class_{c}', 0) for c in range(num_classes)]
axes[1].bar(range(num_classes), class_chis)
axes[1].axhline(y=1.0, color='r', linestyle='--', label='RER=1')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Chi (mean RER)')
axes[1].set_title('Per-Class Difficulty')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Using the Lightning Task

For actual experiments, use the `ClasswiseAETask` with Lightning:

In [ ]:
# Create the Lightning task
task = ClasswiseAETask(
    num_classes=num_classes,
    feature_dim=feature_dim,
    n_components=10,
    hidden_dims=[256],
    learning_rate=0.001,
    umap_loss_weight=0.05,
    use_umap_loss=True,
)

# The task can be used with Lightning Trainer:
# trainer = L.Trainer(max_epochs=50)
# trainer.fit(task, datamodule)

print(f"Task hyperparameters: {task.hparams}")

## 6. CLI Usage

For full experiments, use the CLI:

```bash
# Extract features first
python scripts/extract_features.py --dataset MNIST --model dinov2_vits14 --output ./cache

# Train with config
python src/cli.py fit --config conf/experiment/mnist_dino_simple.yaml

# Test (compute full RER statistics)
python src/cli.py test --config conf/experiment/mnist_dino_simple.yaml --ckpt_path path/to/checkpoint.ckpt
```